# 01. Preprocessing: Column Selection, Time Split and Leakage Fix

CRISP-DM phase: Data preparation

## 1. Setup and Data Loading

### 1.1 Importing Libraries

In [1]:
#necessary imports
import os
import pandas as pd

### 1.2 Loading the Dataset

The full modelling table is team-only because its flood labels contain EM-DAT data, so it is not stored in this repository. The path below is relative to the `notebooks` folder and points to the table in the capstone project folder, which sits next to this repository. `pcode` is read as text so the county codes are kept exactly as they are.

In [2]:
#loading the full modelling table (pcode as text)
df = pd.read_csv('../../Moringa Capstone Project/data/private/model_table_full.csv', dtype={'pcode': str})

In [3]:
#shape of the dataset
df.shape

(14664, 70)

In [4]:
#data type of pcode and number of counties
print(df['pcode'].dtype)
print(df['pcode'].nunique())

str
47


#### Observation

The full table has 14,664 rows and 70 columns, which is 47 counties for 312 months (January 2000 to December 2025). `pcode` is read as text (type `str`) and there are 47 different counties.

## 2. Selecting the 30 Columns

### 2.1 Column Groups

The table has 70 columns and we keep 30 of them: 4 keys, the target, the baseline and the 24 features chosen in the EDA (notebook 04 of the capstone project).

In [5]:
#the four key columns that identify each county-month
keys = ['pcode', 'county', 'year', 'month']

In [6]:
#the target: will the county flood next month?
target = 'flood_next_month'

In [7]:
#the baseline: did the county flood this month? (only for comparison, never a feature)
baseline = 'flood_this_month'

In [8]:
#the 24 selected features
features = ['rain_mm', 'rain_lag1', 'rain_lag2', 'rain_pct_normal', 'neighbour_rain',
            'ndvi_pct_normal', 'oni', 'dmi', 'month_sin', 'month_cos',
            'soil_moisture_l1', 'runoff_rank', 'discharge_rank',
            'pct_flat', 'area_km2', 'lc_tree', 'lc_grass', 'lc_crop', 'lc_bare',
            'pct_within_1km_river', 'pct_water', 'pct_occasional_water', 'pop_density',
            'county_flood_rate']

In [9]:
#number of selected features
len(features)

24

### 2.2 Keeping the 30 Columns

In [10]:
#keeping the keys, the target, the baseline and the 24 features
keep_columns = keys + [target, baseline] + features
df = df[keep_columns]

In [11]:
#shape after keeping the columns
df.shape

(14664, 30)

### 2.3 Checking for Leakage

The target and the baseline must never be features, and neither must the key columns. The next cell lists any feature that is also a key, the target or the baseline, so the list must be empty.

In [12]:
#checking that no key, the target or the baseline is in the features
print(set(features) & set(keys + [target, baseline]))

set()


#### Observation

After keeping the selected columns the table has 14,664 rows and 30 columns: 4 keys, the target, the baseline and the 24 features (`len(features)` prints 24). The leakage check prints an empty set, so none of the features is a key, the target or the baseline. `flood_this_month` is kept only for comparison.

## 3. Selecting the Rows

### 3.1 Missing Values Before Filtering

The next cells show where the missing values are, so the choice of the time period can be checked against the data.

In [13]:
#columns with missing values in the full table
missing = df.isna().sum()
missing[missing > 0]

flood_next_month      47
ndvi_pct_normal     1410
dtype: int64

In [14]:
#years of the rows with no NDVI
df[df['ndvi_pct_normal'].isna()]['year'].value_counts().sort_index()

year
2000    564
2001    564
2002    282
Name: count, dtype: int64

In [15]:
#year and month of the rows with no next-month label
df[df['flood_next_month'].isna()][['year', 'month']].drop_duplicates()

,year,month
311,2025,12


#### Observation

Only two columns have missing values: `ndvi_pct_normal` (1,410 rows) and `flood_next_month` (47 rows). The missing NDVI values are in 2000 (564 rows), 2001 (564 rows) and 2002 (282 rows), which is 30 months (January 2000 to June 2002) for 47 counties, so NDVI starts in July 2002 as expected. The 47 missing labels are all December 2025, one for each county, because there is no January 2026 row to take the label from.

### 3.2 Keeping July 2002 to December 2024

We drop January 2000 to June 2002 because there is no NDVI yet, and all of 2025 because its labels are incomplete.

In [16]:
#start of the period: July 2002 or later
after_start = (df['year'] > 2002) | ((df['year'] == 2002) & (df['month'] >= 7))

In [17]:
#end of the period: December 2024 or earlier
before_end = df['year'] <= 2024

In [18]:
#keeping only the rows inside the period
df = df[after_start & before_end]
df = df.reset_index(drop=True)

In [19]:
#shape after filtering the rows
df.shape

(12690, 30)

In [20]:
#checking the first and the last month kept
print(df['year'].min(), df[df['year'] == df['year'].min()]['month'].min())
print(df['year'].max(), df[df['year'] == df['year'].max()]['month'].max())

2002 7
2024 12


In [21]:
#missing values left after filtering
print(df.isna().sum().sum())

0


In [22]:
#the target is a whole number now that the missing labels are gone
df['flood_next_month'] = df['flood_next_month'].astype(int)

#### Observation

After keeping July 2002 to December 2024 the table has 12,690 rows and 30 columns, which is 270 months for 47 counties. The first month kept is month 7 of 2002 and the last is month 12 of 2024, and no missing values are left. The 1,974 rows dropped are the 1,410 rows from January 2000 to June 2002 and the 564 rows of 2025 (14,664 - 1,974 = 12,690).

### 3.3 Saving the Model Table

`model_table.csv` holds the rows and columns kept above. Its `county_flood_rate` is still the version shipped with the full table, which leaks (see Section 5). It is saved as it is so the table matches the source, and the models use `train.csv` and `test.csv`, not this file.

In [23]:
#creating the data folder if it does not exist yet
os.makedirs('../data', exist_ok=True)

In [24]:
#saving the model table
df.to_csv('../data/model_table.csv', index=False)

## 4. Train/Test Split

### 4.1 Splitting by Time

The split is by time, not random. The aim is to forecast next month from this month, so the model must be tested on months that come after the months it learned from. A random split would mix months from the same years into both sets, which lets the model learn from months right next to the ones it is tested on and makes the results look better than they would be in real use.

- Train: July 2002 to December 2019
- Test: January 2020 to December 2024

In [25]:
#train = July 2002 to December 2019
train = df[df['year'] <= 2019].copy()

In [26]:
#test = January 2020 to December 2024
test = df[df['year'] >= 2020].copy()

### 4.2 Checking the Split

In [27]:
#shapes of the two sets
print(train.shape)
print(test.shape)

(9870, 30)
(2820, 30)


In [28]:
#number of months with no flood (0) and with a flood (1) next month
print(train['flood_next_month'].value_counts())
print(test['flood_next_month'].value_counts())

flood_next_month
0    8988
1     882
Name: count, dtype: int64
flood_next_month
0    2676
1     144
Name: count, dtype: int64


In [29]:
#share of flood rows in each set, in percent
print(round(train['flood_next_month'].mean() * 100, 1))
print(round(test['flood_next_month'].mean() * 100, 1))

8.9
5.1


In [30]:
#last year of the training set and first year of the test set
print(train['year'].max(), test['year'].min())

2019 2020


In [31]:
#number of counties in each set
print(train['pcode'].nunique())
print(test['pcode'].nunique())

47
47


#### Observation

The training set has 9,870 rows (210 months for each of the 47 counties) and the test set has 2,820 rows (60 months for each county). The training set has 882 flood rows (8.9%) and 8,988 non-flood rows, and the test set has 144 flood rows (5.1%) and 2,676 non-flood rows, which matches the numbers we expected. The last training year is 2019 and the first test year is 2020, so every test month comes after every training month, and both sets contain all 47 counties. Floods are rarer in the test period than in the training period, so the test set has only 144 floods to find.

## 5. Fixing the Leak in county_flood_rate

### 5.1 Why It Leaks

The `county_flood_rate` that comes with the table was made from the 2000 to 2019 labels. The training rows are inside that period, so the feature has already seen the training answers. We replace it with the flood rate of each county calculated from the training rows only: the mean of `flood_this_month` for each `pcode` in `train`.

In [32]:
#the shipped county_flood_rate of each county
old_rate = train.groupby('pcode')['county_flood_rate'].first()

In [33]:
#checking the shipped rate is the same on every row of a county (should print 1)
print(train.groupby('pcode')['county_flood_rate'].nunique().max())

1


### 5.2 Recomputing From the Training Rows Only

In [34]:
#flood rate of each county from the training rows only
new_rate = train.groupby('pcode')['flood_this_month'].mean()

In [35]:
#summary of the new rates (47 counties)
new_rate.describe()

count    47.000000
mean      0.089362
std       0.052861
min       0.004762
25%       0.052381
50%       0.090476
75%       0.123810
max       0.223810
Name: flood_this_month, dtype: float64

In [36]:
#how far the shipped rate is from the new rate, and how strongly they are related
print(round((old_rate - new_rate).abs().mean(), 4))
print(round((old_rate - new_rate).abs().max(), 4))
print(round(old_rate.corr(new_rate), 3))

0.0104
0.0308
0.997


### 5.3 Mapping the Rate Onto Train and Test

The same value of each county is put on both sets. The test set does not get its own rate, because that would use the test answers.

In [37]:
#replacing county_flood_rate in train and test with the training-only rate
train['county_flood_rate'] = train['pcode'].map(new_rate)
test['county_flood_rate'] = test['pcode'].map(new_rate)

In [38]:
#checking that no county is missing a rate
print(train['county_flood_rate'].isna().sum())
print(test['county_flood_rate'].isna().sum())

0
0


In [39]:
#checking that every county has the same rate in train and test
print((train.groupby('pcode')['county_flood_rate'].first() == test.groupby('pcode')['county_flood_rate'].first()).all())

True


#### Observation

The shipped `county_flood_rate` has one value for each county (the check prints 1). The new training-only rates range from 0.005 to 0.224 across the 47 counties, with a mean of 0.089, which is close to the 8.9% flood share of the training set. The shipped rate is on average 0.010 away from the new rate (at most 0.031) and the two are strongly related (correlation 0.997), so the fix changes the values only a little, but the new rate is the only one that has not seen the test years. No county is missing a rate in train or in test (0 and 0), and the last check prints True, so each county has the same rate in both sets. One small limit remains: each training row's own outcome is one of the 210 months used to calculate the rate of its county, so it has a very small part (about 0.5%) in that rate. The test rows are not touched by this.

## 6. Missing Values and Scaling

Nothing is imputed or scaled in this notebook. Scaling goes inside the pipeline of each model notebook, so it is learned from the training rows only and the test set never influences it. The next cell only checks that there are no missing values to deal with.

In [40]:
#missing values in train and in test
print(train.isna().sum().sum())
print(test.isna().sum().sum())

0
0


#### Observation

Both sets have 0 missing values, so nothing needs to be imputed here, and the features are left unscaled for the pipeline in the model notebooks.

## 7. Saving the Split

The training and test sets are saved with all 30 columns, so every model notebook loads the same files. The models use the 24 features and the target, and the baseline column `flood_this_month` is kept only to compare against.

In [41]:
#saving the training and test sets
train.to_csv('../data/train.csv', index=False)
test.to_csv('../data/test.csv', index=False)

In [42]:
#checking the saved files (pcode as text)
train_check = pd.read_csv('../data/train.csv', dtype={'pcode': str})
test_check = pd.read_csv('../data/test.csv', dtype={'pcode': str})
print(train_check.shape)
print(test_check.shape)

(9870, 30)
(2820, 30)


In [43]:
#checking the two files have the same columns and no missing values
print(list(train_check.columns) == list(test_check.columns))
print(train_check.isna().sum().sum() + test_check.isna().sum().sum())

True
0


In [44]:
#columns of the saved training file
print(train_check.columns.tolist())

['pcode', 'county', 'year', 'month', 'flood_next_month', 'flood_this_month', 'rain_mm', 'rain_lag1', 'rain_lag2', 'rain_pct_normal', 'neighbour_rain', 'ndvi_pct_normal', 'oni', 'dmi', 'month_sin', 'month_cos', 'soil_moisture_l1', 'runoff_rank', 'discharge_rank', 'pct_flat', 'area_km2', 'lc_tree', 'lc_grass', 'lc_crop', 'lc_bare', 'pct_within_1km_river', 'pct_water', 'pct_occasional_water', 'pop_density', 'county_flood_rate']


In [45]:
#flood rows in the saved files
print(train_check['flood_next_month'].sum())
print(test_check['flood_next_month'].sum())

882
144


#### Observation

`train.csv` has 9,870 rows and 30 columns and `test.csv` has 2,820 rows and 30 columns. Both files have the same columns in the same order (True) and no missing values (0), and they hold 882 and 144 flood rows, so they match the split in Section 4. The 30 columns are the 4 keys, `flood_next_month`, `flood_this_month` and the 24 features, ending with `county_flood_rate`. The next notebook, `LogisticRegression_Base.ipynb`, trains the base model on these files.

## 8. Summary of Preprocessing

- **Columns:** 30 kept from 70: 4 keys, the target `flood_next_month`, the baseline `flood_this_month` and 24 features. The baseline is never a feature.
- **Rows:** July 2002 to December 2024 (12,690 rows, 47 counties). January 2000 to June 2002 has no NDVI and 2025 has incomplete labels.
- **Split by time:** train July 2002 to December 2019 (9,870 rows, 882 floods, 8.9%) and test January 2020 to December 2024 (2,820 rows, 144 floods, 5.1%).
- **Leak fixed:** `county_flood_rate` is now the flood rate of each county from the training rows only, with the same value on train and test.
- **Not done here:** no scaling and no imputing (there are no missing values). Scaling goes inside the model pipelines.
- **Saved:** `data/model_table.csv`, `data/train.csv` and `data/test.csv`. None of them is committed, because of the EM-DAT licence.

The next notebook, `LogisticRegression_Base.ipynb`, trains the base model on `train.csv` and compares it with the persistence baseline on `test.csv`.